# 02 — EDA and Validation Design

Builds the validation infrastructure the rest of the project depends on, and answers a
scoped set of high-value questions about the real preprocessed data (`notebooks/
01_preprocessing_pipeline.ipynb` must have been run first -- this notebook only loads its
artifacts, it never re-derives them).

Produces two permanent artifacts:

- `data/processed/connectivity_folds.parquet` -- primary CV, one row per `connectivity_key`
- `data/processed/scaffold_folds.parquet` -- harder secondary CV, one row per Murcko scaffold

Then covers: dataset structure, molecular diversity, formula/mass ambiguity, structural
neighbor density, and train/test distribution shift -- with actual quantitative metrics
(Wasserstein / KS / Jensen-Shannon), not eyeballed histograms.

In [1]:
import sys
from pathlib import Path

# --- src/ imports (repo-relative; works whether the notebook is launched from repo root or
#     from notebooks/) -------------------------------------------------------------------------
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import pandas as pd

from casmi.paths import get_project_paths
from casmi.io.artifacts import load_artifact, save_artifact
from casmi.validation.folds import build_connectivity_folds, build_scaffold_folds, check_group_leakage, fold_summary
from casmi.validation.shift import shift_report
from casmi.chemistry.similarity import morgan_fingerprints, nearest_structure_neighbors
from casmi.candidates.mass_index import MassIndex

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
SEED = 42

## 0. Load preprocessing artifacts

In [2]:
paths = get_project_paths()

molecule_metadata = load_artifact("molecule_metadata")
train_meta = load_artifact("train_spectrum_metadata")
test_meta = load_artifact("test_spectrum_metadata")
test_features = load_artifact("test_spectrum_features")
structure_table = pd.read_parquet(paths.interim / "structure_table.parquet")

print(f"molecule_metadata      {molecule_metadata.shape}")
print(f"train_spectrum_metadata{train_meta.shape}")
print(f"test_spectrum_metadata {test_meta.shape}")
print(f"structure_table        {structure_table.shape}")

molecule_metadata      (274195, 8)
train_spectrum_metadata(2539608, 27)
test_spectrum_metadata (1213, 16)
structure_table        (277566, 28)


## 1. Connectivity folds (primary CV)

`GroupKFold` at the spectrum-row level: fold size is balanced by SPECTRUM count, while every
`connectivity_key` is guaranteed to land entirely inside one fold. Saved as a permanent
artifact -- every later notebook reads the fold assignment from here rather than
re-deriving it.

In [3]:
connectivity_folds = build_connectivity_folds(train_meta, n_splits=5)
leakage_check = check_group_leakage(train_meta, connectivity_folds)
print(f"[{'PASS' if leakage_check.passed else 'FAIL'}] {leakage_check.name}: {leakage_check.detail}")
assert leakage_check.passed

display(fold_summary(train_meta, connectivity_folds))

save_artifact(
    connectivity_folds, "connectivity_folds", paths.processed,
    input_fingerprints={}, description="Primary CV: one row per connectivity_key -> fold (0-4), GroupKFold balanced by spectrum count",
)
print(f"saved {len(connectivity_folds)} connectivity_key -> fold assignments")

[PASS] no connectivity_key leakage across folds: no leakage


,fold,n_spectra,n_groups
0,0,507922,54839
1,1,507922,54839
2,2,507922,54839
3,3,507921,54839
4,4,507921,54839


saved 274195 connectivity_key -> fold assignments


## 2. Scaffold folds (harder secondary CV)

Same algorithm, grouped by Murcko scaffold instead -- many connectivities share one scaffold,
so this is a much coarser split that stress-tests generalization to genuinely new chemistry
rather than interpolation within an already-seen scaffold family.

In [4]:
struct_per_key = structure_table.drop_duplicates("connectivity_key").set_index("connectivity_key")
connectivity_to_scaffold = struct_per_key["scaffold_smiles"]

scaffold_folds = build_scaffold_folds(train_meta, connectivity_to_scaffold, n_splits=5)
train_meta_with_scaffold = train_meta.assign(scaffold_smiles=train_meta["connectivity_key"].map(connectivity_to_scaffold))
display(fold_summary(train_meta_with_scaffold, scaffold_folds, group_col="scaffold_smiles"))

n_scaffolds = struct_per_key["scaffold_smiles"].nunique()
n_connectivity = len(struct_per_key)
print(f"{n_scaffolds:,} unique scaffolds vs {n_connectivity:,} unique connectivity_key "
      f"({n_connectivity / n_scaffolds:.1f} connectivities/scaffold on average)")

save_artifact(
    scaffold_folds, "scaffold_folds", paths.processed,
    input_fingerprints={}, description="Secondary/harder CV: one row per scaffold_smiles -> fold (0-4)",
)

,fold,n_spectra,n_groups
0,0,507922,9788
1,1,507922,9789
2,2,507922,9795
3,3,507921,9795
4,4,507921,9795


48,962 unique scaffolds vs 274,195 unique connectivity_key (5.6 connectivities/scaffold on average)


WindowsPath('C:/Users/myben/OneDrive/Documents/Enveda/data/processed/scaffold_folds.parquet')

## 3. Molecule-level table

One row per `connectivity_key`, joining `molecule_metadata` (spectrum-derived) with structural
descriptors (`structure_table`, deduplicated to one row per key -- tautomer SMILES sharing a
`connectivity_key` are collapsed by taking the first) and `molecular_formula` (read off
`train_spectrum_metadata`, constant within a connectivity_key by construction).

In [5]:
mol = molecule_metadata.merge(struct_per_key.reset_index(), on="connectivity_key", how="left")
mol = mol.merge(
    train_meta[["connectivity_key", "molecular_formula"]].drop_duplicates("connectivity_key"),
    on="connectivity_key", how="left",
)
print(f"mol: {mol.shape}, {mol['exact_mass'].notna().mean():.1%} with exact_mass")
mol.describe(include="number").T

mol: (274195, 36), 100.0% with exact_mass


,count,mean,std,min,25%,50%,75%,max
n_spectra,274195.0,9.262051,21.329442,1.000000,4.00000,6.00000,9.000000,1.468000e+03
n_adducts,274195.0,1.884134,1.067154,1.000000,1.00000,2.00000,2.000000,1.500000e+01
n_instruments,274195.0,1.166199,0.899947,0.000000,1.00000,1.00000,1.000000,2.900000e+01
n_ion_modes,274195.0,1.348540,0.476509,1.000000,1.00000,1.00000,2.000000,2.000000e+00
precursor_mz_min,274195.0,359.825820,126.037549,2.000000,305.11660,334.16830,368.208800,2.679480e+03
precursor_mz_max,274195.0,474.287992,4644.376243,2.000000,320.09510,374.13910,649.209000,2.430333e+06
precursor_mz_range,274195.0,114.462172,4643.287743,0.000000,0.00000,2.01470,307.129100,2.430090e+06
exact_mass,274195.0,357.832127,132.909598,31.042199,304.16994,333.14887,365.193277,3.399573e+03
molecular_weight,274195.0,358.181073,132.996974,31.058000,304.41500,333.43200,365.764500,3.401902e+03
num_atoms,274195.0,48.922457,22.998315,4.000000,39.00000,44.00000,50.000000,4.640000e+02


## 4. Dataset structure

Spectra and adducts per connectivity_key, plus instrument/ion-mode coverage.

In [6]:
print("spectra per connectivity_key:")
display(molecule_metadata["n_spectra"].describe())

print("\nadducts per connectivity_key:")
display(molecule_metadata["n_adducts"].describe())

print("\ninstrument_type counts (train spectra):")
display(train_meta["instrument_type"].value_counts())

print("\nionization_mode counts (train spectra):")
display(train_meta["ionization_mode"].value_counts())

spectra per connectivity_key:


count    274195.000000
mean          9.262051
std          21.329442
min           1.000000
25%           4.000000
50%           6.000000
75%           9.000000
max        1468.000000
Name: n_spectra, dtype: float64


adducts per connectivity_key:


count    274195.000000
mean          1.884134
std           1.067154
min           1.000000
25%           1.000000
50%           2.000000
75%           2.000000
max          15.000000
Name: n_adducts, dtype: float64


instrument_type counts (train spectra):


instrument_type
timsTOF          1154969
Orbitrap          745562
QTOF              125936
LC-ESI-QTOF       104482
ESI-QFT            77342
                  ...   
Q-Exactive             1
QIT-FT                 1
SI-BE                  1
Thermo LTQ             1
Waters SYNAPT          1
Name: count, Length: 82, dtype: int64


ionization_mode counts (train spectra):


ionization_mode
positive    1899640
negative     639968
Name: count, dtype: int64

## 5. Molecular diversity

Exact mass, ring/heteroatom composition, and scaffold coverage across the unique train
structure library.

In [7]:
display(mol[["exact_mass", "num_rings", "num_aromatic_rings", "num_heteroatoms", "num_rotatable_bonds", "tpsa", "logp"]].describe())

scaffold_counts = mol["scaffold_smiles"].value_counts()
print(f"\nlargest scaffold families (by connectivity_key count):")
display(scaffold_counts.head(10))
print(f"\n{(scaffold_counts == 1).sum():,} / {len(scaffold_counts):,} scaffolds are singletons (exactly one connectivity_key)")

,exact_mass,num_rings,num_aromatic_rings,num_heteroatoms,num_rotatable_bonds,tpsa,logp
count,274195.000000,274195.000000,274195.000000,274195.000000,274195.000000,274195.000000,274195.000000
mean,357.832127,2.812703,1.726468,6.763909,5.972221,77.068838,2.715348
std,132.909598,1.225460,0.978894,3.061914,6.954847,47.948611,2.417636
min,31.042199,0.000000,0.000000,0.000000,0.000000,0.000000,-26.462500
25%,304.169940,2.000000,1.000000,5.000000,3.000000,56.590000,1.680800
50%,333.148870,3.000000,2.000000,6.000000,4.000000,69.560000,2.476800
75%,365.193277,3.000000,2.000000,8.000000,6.000000,84.860000,3.282500
max,3399.572640,32.000000,32.000000,88.000000,86.000000,1405.370000,26.795300



largest scaffold families (by connectivity_key count):


scaffold_smiles
                          13874
C1CCCCC1                  10456
C1CCC(C2CCCC2)CC1          2968
C1CCC2CCCC2C1              2758
CC(CC1CCCCC1)C1CCCCC1      2654
C1CCCC1                    2560
CC(CCC1CCCCC1)C1CCCCC1     2424
C1CCC(CC2CCCCC2)CC1        2299
C1CCC(C2CCCCC2)CC1         1779
CC(CCC1CCCCC1)C1CCCC1      1691
Name: count, dtype: int64


30,585 / 48,962 scaffolds are singletons (exactly one connectivity_key)


## 6. Formula ambiguity

> Given the molecular formula alone, how many distinct connectivities remain plausible?

In [8]:
formula_ambiguity = mol.dropna(subset=["molecular_formula"]).groupby("molecular_formula")["connectivity_key"].nunique()
display(formula_ambiguity.describe())

print("\nmost ambiguous formulas (most co-eluting isomers):")
display(formula_ambiguity.sort_values(ascending=False).head(10))

count    51197.000000
mean         5.355685
std         15.805302
min          1.000000
25%          1.000000
50%          1.000000
75%          4.000000
max        477.000000
Name: connectivity_key, dtype: float64


most ambiguous formulas (most co-eluting isomers):


molecular_formula
C19H24N4O2    477
C18H24N4O2    463
C19H26N4O2    422
C17H22N4O2    409
C18H22N4O2    380
C18H23N3O2    354
C18H23N3O3    350
C19H23N3O3    349
C20H25N3O2    348
C17H21N3O2    342
Name: connectivity_key, dtype: int64

## 7. Mass ambiguity

> Given the exact neutral mass alone (no formula), how many structures fall within a
> realistic MS precursor-error tolerance? Built on `MassIndex` (`casmi.candidates.mass_index`)
> -- the same index later notebooks use for real candidate generation.

In [9]:
mass_index = MassIndex.from_molecule_table(mol)
print(f"MassIndex built over {len(mass_index):,} structures")

rng = np.random.RandomState(SEED)
sample_masses = mol["exact_mass"].dropna().sample(min(5000, mol["exact_mass"].notna().sum()), random_state=SEED).to_numpy()

rows = []
for ppm in [1, 2, 3, 5, 10, 20, 50]:
    counts = np.array([mass_index.candidate_count(m, tolerance_ppm=ppm) for m in sample_masses])
    rows.append({
        "tolerance_ppm": ppm, "median": np.median(counts), "p75": np.percentile(counts, 75),
        "p90": np.percentile(counts, 90), "p95": np.percentile(counts, 95),
        "p99": np.percentile(counts, 99), "max": int(counts.max()),
    })
mass_ambiguity_grid = pd.DataFrame(rows)
display(mass_ambiguity_grid)

MassIndex built over 274,195 structures


,tolerance_ppm,median,p75,p90,p95,p99,max
0,1,20.0,69.00,149.0,208.00,349.01,477
1,2,21.0,70.00,149.0,208.00,349.01,477
2,3,25.0,72.00,150.0,208.10,349.01,480
3,5,36.0,89.00,159.1,220.00,352.00,482
4,10,57.0,124.00,200.1,272.00,389.00,500
5,20,94.0,184.25,291.0,347.05,430.00,514
6,50,240.0,476.25,683.0,759.00,882.00,977


## 8. Structural neighborhood

Nearest-neighbor Tanimoto similarity over a random subsample of the structure library (all-pairs
similarity is O(n^2) -- see `casmi.chemistry.similarity`'s docstring; a full run over ~274k
structures is not attempted here). Tells us whether the library is mostly isolated singletons or
has dense structural families -- directly relevant to later hard-negative mining.

In [10]:
SIMILARITY_SAMPLE_SIZE = 3000
sub = mol.dropna(subset=["canonical_smiles"]).sample(min(SIMILARITY_SAMPLE_SIZE, len(mol)), random_state=SEED)
fps = morgan_fingerprints(sub["canonical_smiles"].tolist())
neighbors = nearest_structure_neighbors(sub["connectivity_key"].tolist(), fps, top_k=5)

display(neighbors["nn_tanimoto"].describe())
print(f"\n{(neighbors['nn_tanimoto'] >= 0.7).mean():.1%} of sampled structures have a close analogue (nn_tanimoto >= 0.7)")
print(f"{(neighbors['nn_tanimoto'] < 0.3).mean():.1%} of sampled structures are relatively isolated (nn_tanimoto < 0.3)")

count    3000.000000
mean        0.415860
std         0.138499
min         0.100000
25%         0.333333
50%         0.383333
75%         0.457627
max         1.000000
Name: nn_tanimoto, dtype: float64


4.9% of sampled structures have a close analogue (nn_tanimoto >= 0.7)
11.9% of sampled structures are relatively isolated (nn_tanimoto < 0.3)


## 9. Train/test distribution shift

Quantitative shift metrics (`casmi.validation.shift`), not histograms: KS statistic (bounded,
comparable across columns) + Wasserstein distance (column's own units) for numeric fields,
Jensen-Shannon distance for categorical fields. `test_meta` has no `num_peaks` column (unlike
train), so it's joined from `test_spectrum_features` (computed during preprocessing).

In [11]:
test_meta_aug = test_meta.merge(test_features[["spectrum_id", "n_peaks"]], on="spectrum_id", how="left")
test_meta_aug = test_meta_aug.rename(columns={"n_peaks": "num_peaks"})

numeric_report, categorical_report = shift_report(
    train_meta, test_meta_aug,
    numeric_cols=["precursor_mz", "num_peaks", "ce_mean"],
    categorical_cols=["adduct", "ionization_mode", "instrument_type"],
)
print("numeric shift (sorted by KS statistic, largest first):")
display(numeric_report)
print("\ncategorical shift (sorted by Jensen-Shannon distance, largest first):")
display(categorical_report)

numeric shift (sorted by KS statistic, largest first):


,name,n_train,n_test,ks_stat,ks_pvalue,wasserstein
0,num_peaks,2539608,1213,0.425790,6.442044e-200,188.917929
1,precursor_mz,2539608,1213,0.319224,2.386540e-110,106.742398
2,ce_mean,2202165,1213,0.287836,2.153535e-89,8.306907



categorical shift (sorted by Jensen-Shannon distance, largest first):


,name,n_train,n_test,n_categories_train,n_categories_test,n_categories_test_only,jensen_shannon_distance
0,instrument_type,2506555,1213,82,1,0,0.585770
1,adduct,2539608,1213,121,7,0,0.344340
2,ionization_mode,2539608,1213,2,2,0,0.067526


## Summary

If every check above ran cleanly, `data/processed/connectivity_folds.parquet` and
`data/processed/scaffold_folds.parquet` are ready for any later notebook to load directly
(`load_artifact("connectivity_folds")` / `load_artifact("scaffold_folds")`) rather than
rebuilding fold assignments inline.

Notebook 03 (high-recall candidate generation) should optimize **candidate recall**, not MRR --
the mass-ambiguity grid above (Section 7) sets the realistic ceiling on how tight a ppm
tolerance can be before the true candidate starts falling outside the window.